
# 🧠 Agente de Carreira Adaptativa (ACA) – GS Prompt and Artificial Intelligence

**Integrantes do Grupo:**  
- Lucas Andrade Souza – RM 564066  
- Luis Otavio Santini Feitosa – RM 566556  

**Curso:** Ciência da Computação  
**Disciplina:** Prompt and Artificial Intelligence  
**Professor:** Andrei  
**Modelo de LLM:** GPT-4o-mini  
**Ferramenta:** OpenAI API (via `requests`)

---

Este notebook implementa todas as **4 funcionalidades (F1 a F4)** exigidas no projeto **Agente de Carreira Adaptativa (ACA)**:

1. **F1:** Análise de Perfil e Risco de Automação  
2. **F2:** Plano de Upskilling Personalizado  
3. **F3:** Sugestão de Caminho de Reskilling  
4. **F4:** Simulação de Entrevista e Avaliação  

Cada seção contém prompts específicos, integração com a API e saída em JSON conforme solicitado.

---


In [ ]:

# ============================================================
# 🔧 CONFIGURAÇÃO INICIAL
# ============================================================
# Execute esta célula primeiro!
# Aqui você deve inserir sua chave de API da OpenAI para permitir o uso do LLM.

import time
import json
import requests

API_KEY = input("Cole sua chave OpenAI (sk-proj-...): ").strip()
BASE_URL = "https://api.openai.com/v1/chat/completions"
MODEL = "gpt-4o-mini"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

def post_chat(payload, timeout=60):
    """Função que envia requisições à API com controle de erros e retries."""
    max_retries = 6
    backoff = 4
    last_exc = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(BASE_URL, headers=headers, json=payload, timeout=timeout)
            status = response.status_code
            print(f"🔹 Requisição enviada (tentativa {attempt}), status: {status}")

            if status == 200:
                return response.json()
            elif status == 429:
                wait = backoff * attempt
                print(f"⚠️ 429 recebido — aguardando {wait:.1f}s")
                time.sleep(wait)
                continue
            else:
                print(f"❌ Erro {status}: {response.text}")
                break
        except Exception as e:
            last_exc = e
            print(f"⚠️ Erro na tentativa {attempt}: {e}")
            time.sleep(backoff)

    if last_exc:
        raise last_exc
    raise Exception("Erro desconhecido na requisição POST")

def extract_json_block(text):
    """Extrai o primeiro bloco JSON válido de uma resposta do modelo."""
    try:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start != -1 and end != -1:
            return json.loads(text[start:end])
    except Exception:
        pass
    return {"texto": text, "status_json": "falha"}


In [ ]:
# ============================================================
# 🏆 AGENTE DE CARREIRA ADAPTATIVA (ACA) - VERSÃO CORRIGIDA 🏆
# ============================================================
# Implementa a Engenharia de Prompt focada em JSON e Lógica F1-F4.

import time
import json
import requests

# =======================================================
# 1️⃣ - CONFIGURAÇÃO DA API
# =======================================================
# OBS: Sua chave anterior foi removida por segurança. Cole-a novamente.
API_KEY = input("Cole sua chave OpenAI (sk-proj-...): ").strip()
BASE_URL = "https://api.openai.com/v1/chat/completions"
MODEL = "gpt-4o-mini" # Mantido o modelo leve e eficiente

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# =======================================================
# 2️⃣ - FUNÇÃO DE REQUISIÇÃO COM TRATAMENTO DE ERROS
# =======================================================
def post_chat(payload, timeout=60):
    max_retries = 6
    backoff = 4
    last_exc = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(BASE_URL, headers=headers, json=payload, timeout=timeout)
            status = response.status_code
            print(f"🔹 Requisição enviada (tentativa {attempt}), status: {status}")

            if status == 200:
                return response.json()
            elif status == 429:
                wait = backoff * attempt
                print(f"⚠️ 429 recebido — aguardando {wait:.1f}s")
                time.sleep(wait)
                continue
            else:
                print(f"❌ Erro {status}: {response.text}")
                break
        except Exception as e:
            last_exc = e
            print(f"⚠️ Erro na tentativa {attempt}: {e}")
            time.sleep(backoff)

    if last_exc:
        raise last_exc
    raise Exception("Erro desconhecido na requisição POST")

# =======================================================
# 3️⃣ - FUNÇÃO AUXILIAR PARA EXTRAR JSON DO RETORNO
# =======================================================
def extract_json_block(text):
    """Tenta extrair o primeiro e único bloco JSON de uma string."""
    try:
        # Encontra a primeira e a última chave, adicionando 1 para incluir '}'
        start = text.find("{")
        end = text.rfind("}") + 1
        if start != -1 and end != -1:
            return json.loads(text[start:end])
    except Exception as e:
        # Se falhar, retorna o texto bruto para debug.
        # print(f"Erro ao extrair JSON: {e}")
        pass
    return {"texto": text, "status_json": "falha"}

# =======================================================
# 4️⃣ - FASE 1: ANÁLISE DE PERFIL E RISCO DE AUTOMAÇÃO (F1)
# =======================================================
def run_F1(profissao_ficticia, tarefas):
    print("\n------ F1: Análise de Risco ------")
    
    system_prompt_f1 = f"""
    Você é um Analista de Risco de Automação e Futuro do Trabalho.
    Sua única tarefa é analisar a profissão '{profissao_ficticia}' e suas tarefas principais, e gerar uma análise em JSON.

    REGRAS DE SAÍDA OBRIGATÓRIAS:
    1. Responda APENAS com o bloco JSON, sem texto introdutório.
    2. Estime o risco_automacao_percentual como um número inteiro de 0 a 100.
    3. A lista de habilidades deve ter EXATAMENTE 3 itens em cada categoria.

    FORMATO JSON OBRIGATÓRIO:
    {{
      "fase": "F1",
      "profissao_analisada": "{profissao_ficticia}",
      "risco_automacao_percentual": 0,
      "justificativa_risco": "Curta justificativa sobre a estimativa.",
      "habilidades_criticas_hard": ["Hard Skill 1", "Hard Skill 2", "Hard Skill 3"],
      "habilidades_criticas_soft": ["Soft Skill 1", "Soft Skill 2", "Soft Skill 3"]
    }}
    """
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt_f1},
            {"role": "user", "content": f"Analise o risco de automação e as habilidades críticas para a profissão: {profissao_ficticia}. Tarefas principais: {tarefas}."}
        ]
    }
    response = post_chat(payload)
    content = response["choices"][0]["message"]["content"]
    resultado = extract_json_block(content)
    # Se a extração falhar, o código de relatório tratará
    return resultado

# =======================================================
# 5️⃣ - FASE 2: PLANO DE UPSKILLING PERSONALIZADO (F2)
# =======================================================
def run_F2(res_f1_json):
    print("\n------ F2: Plano de Upskilling ------")
    
    # Extrai as habilidades críticas da F1 para o prompt
    hard_skills = res_f1_json.get("habilidades_criticas_hard", [])
    soft_skills = res_f1_json.get("habilidades_criticas_soft", [])
    habilidades_foco = hard_skills + soft_skills

    system_prompt_f2 = """
    Você é um Consultor de Upskilling (Aprimoramento) especializado.
    Sua única tarefa é criar um plano de aprimoramento (Upskilling) e sugerir um recurso de aprendizado (curso/certificação) para cada área.

    REGRAS DE SAÍDA OBRIGATÓRIAS:
    1. Responda APENAS com o bloco JSON.
    2. Gere EXATAMENTE 5 sugestões de Upskilling.

    FORMATO JSON OBRIGATÓRIO:
    {
      "fase": "F2",
      "plano_upskilling": [
        {
          "area_aprimoramento": "Área de aprimoramento 1",
          "recurso_sugerido": "Curso/Certificação/Livro 1"
        },
        // ... mais 4 objetos
      ]
    }
    """
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt_f2},
            {"role": "user", "content": f"Crie um plano de Upskilling para um profissional que precisa desenvolver estas habilidades críticas e de futuro: {habilidades_foco}."}
        ]
    }
    response = post_chat(payload)
    content = response["choices"][0]["message"]["content"]
    resultado = extract_json_block(content)
    resultado["fase"] = "F2"
    return resultado

# =======================================================
# 6️⃣ - FASE 3: SUGESTÃO DE CAMINHO DE RESKILLING (F3)
# =======================================================
def run_F3(profissao_origem, area_interesse):
    print("\n------ F3: Mapeamento de Reskilling ------")
    
    system_prompt_f3 = """
    Você é um Mapeador de Reskilling (Transição de Carreira) especializado.
    Sua única tarefa é analisar a transição de '{profissao_origem}' para '{area_interesse}', identificando habilidades transferíveis e as 5 principais novas habilidades (Reskilling) necessárias.

    REGRAS DE SAÍDA OBRIGATÓRIAS:
    1. Responda APENAS com o bloco JSON.
    2. A lista de novas habilidades deve ter EXATAMENTE 5 itens.

    FORMATO JSON OBRIGATÓRIO:
    {{
      "fase": "F3",
      "transicao": "De {profissao_origem} para {area_interesse}",
      "habilidades_transferiveis": ["Transferível 1", "Transferível 2", "Transferível 3"],
      "novas_habilidades_reskilling": ["Nova Habilidade 1", "Nova Habilidade 2", "Nova Habilidade 3", "Nova Habilidade 4", "Nova Habilidade 5"]
    }}
    """
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt_f3.format(profissao_origem=profissao_origem, area_interesse=area_interesse)},
            {"role": "user", "content": f"Mapeie a transição de carreira de {profissao_origem} para {area_interesse}."}
        ]
    }
    response = post_chat(payload)
    content = response["choices"][0]["message"]["content"]
    resultado = extract_json_block(content)
    resultado["fase"] = "F3"
    return resultado

# =======================================================
# 7️⃣ - FASE 4: SIMULAÇÃO DE ENTREVISTA (AVALIAÇÃO) (F4)
# =======================================================
def run_F4(area_reskilling, respostas_usuario):
    print("\n------ F4: Simulação de Entrevista ------")
    
    perguntas = [
        "Descreva uma situação onde você usou análise de dados para resolver um problema complexo.",
        "Quais ferramentas de análise você domina e como elas o ajudaram a tomar decisões estratégicas?",
        "Qual é o seu maior desafio na área de Reskilling e como você planeja superá-lo?"
    ]
    
    resultado = {"Perguntas_e_Avaliacoes": [], "Feedback_Final": None, "fase": "F4"}

    system_prompt_avaliacao = """
    Você é um Avaliador de Entrevistas especializado na área de {area_reskilling}.
    Sua única tarefa é **avaliar a resposta do candidato** para a pergunta, com base estrita nos critérios: Clareza, Relevância e Profundidade.

    REGRAS DE SAÍDA OBRIGATÓRIAS:
    1. Responda APENAS com o bloco JSON, sem texto introdutório.
    2. As pontuações (clareza/relevancia/profundidade) devem ser 'Ótimo', 'Bom', 'Mediano' ou 'Fraco'.

    FORMATO JSON OBRIGATÓRIO:
    {{
      "pergunta_simulada": "A pergunta que você está avaliando.",
      "resposta_candidato": "A resposta que o candidato deu.",
      "avaliacao_detalhada": "Análise detalhada de como a resposta se saiu nos 3 critérios.",
      "pontuacao_clareza": "Ótimo",
      "pontuacao_relevancia": "Bom",
      "pontuacao_profundidade": "Mediano"
    }}
    """
    
    for i, pergunta in enumerate(perguntas):
        print(f"⏳ Aguardando 10s antes da pergunta {i+1}...")
        time.sleep(10)
        
        user_response = respostas_usuario[i]

        payload = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_prompt_avaliacao.format(area_reskilling=area_reskilling)},
                {"role": "user", "content": f"Pergunta de Entrevista: {pergunta}\\nResposta do candidato (simulada): {user_response}"}
            ]
        }
        response = post_chat(payload)
        content = response["choices"][0]["message"]["content"]
        resultado["Perguntas_e_Avaliacoes"].append(extract_json_block(content))

    # Feedback final
    print("⏳ Aguardando 10s antes do feedback final...")
    time.sleep(10)

    # Prompt para o feedback final (que resume a entrevista)
    payload_final = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "Gere um Feedback Final de 3 parágrafos sobre a Simulação de Entrevista e o caminho de Reskilling. Não use JSON, use Markdown."},
            {"role": "user", "content": f"Gere o feedback final para o candidato que respondeu às perguntas sobre a área de {area_reskilling}. As avaliações anteriores foram: {resultado['Perguntas_e_Avaliacoes']}"}
        ]
    }
    response_final = post_chat(payload_final)
    content_final = response_final["choices"][0]["message"]["content"]
    resultado["Feedback_Final"] = {"texto": content_final}
    
    return resultado

# =======================================================
# 8️⃣ - EXECUÇÃO COMPLETA (F1 → F4)
# =======================================================

# Dados fictícios para a execução:
# F1: Analisa uma profissão (Analista de Marketing)
PROF_F1 = "Analista de Marketing Digital"
TAREFAS_F1 = "Gestão de campanhas pagas, criação de relatórios de performance em Excel, organização de dados de CRM."
# F3/F4: Transição/Entrevista para Data Analytics
PROF_ORIGEM_F3 = PROF_F1
AREA_INTERESSE_F3 = "Analista de Dados (foco em Business Intelligence)"
# F4: Respostas simuladas do usuário (agora mais concretas para avaliação)
RESPOSTAS_F4 = [
    "Identifiquei que o custo de aquisição (CAC) estava inflacionado devido a um segmento de anúncios, o que me fez realocar 30% do orçamento para canais mais eficientes, salvando R$ 10.000 em um mês.",
    "Domino Excel avançado e comecei a usar Power BI para criar dashboards interativos. Estas ferramentas permitiram visualização rápida dos KPIs, o que não era possível com relatórios estáticos.",
    "O maior desafio é a falta de experiência em Python e SQL avançado, mas estou fazendo o curso 'Python para Data Science' e praticando em bases de dados públicas para compensar."
]


# 1. Executa F1
res_f1 = run_F1(PROF_F1, TAREFAS_F1)
# 2. Executa F2 (usa o JSON da F1)
res_f2 = run_F2(res_f1)
# 3. Executa F3
res_f3 = run_F3(PROF_ORIGEM_F3, AREA_INTERESSE_F3)
# 4. Executa F4
res_f4 = run_F4(AREA_INTERESSE_F3, RESPOSTAS_F4)


# =======================================================
# 9️⃣ - FUNÇÃO DE RELATÓRIO FINAL VISUAL
# =======================================================
def print_relatorio_final(res_f1, res_f2, res_f3, res_f4):
    print("\n\n" + "="*90)
    print("🏆 RELATÓRIO DE EVOLUÇÃO PROFISSIONAL - AGENTE DE CARREIRA ADAPTATIVA (ACA) 🏆")
    print("="*90 + "\n")

    # F1
    print("📘 FASE 1: ANÁLISE DE PERFIL E RISCO DE AUTOMAÇÃO (F1)")
    print("-" * 90)
    print(json.dumps(res_f1, indent=2, ensure_ascii=False))
    print("\n")

    # F2
    print("💡 FASE 2: PLANO DE UPSKILLING PERSONALIZADO (F2)")
    print("-" * 90)
    print(json.dumps(res_f2, indent=2, ensure_ascii=False))
    print("\n")

    # F3
    print("🚀 FASE 3: SUGESTÃO DE CAMINHO DE RESKILLING (F3)")
    print("-" * 90)
    print(json.dumps(res_f3, indent=2, ensure_ascii=False))
    print("\n")

    # F4 - Avaliações
    print("🧭 FASE 4: SIMULAÇÃO DE ENTREVISTA E AVALIAÇÃO (F4)")
    print("-" * 90)

    for i, avaliacao in enumerate(res_f4.get("Perguntas_e_Avaliacoes", []), start=1):
        print(f"\n🔹 AVALIAÇÃO DA PERGUNTA {i}:")
        print(json.dumps(avaliacao, indent=2, ensure_ascii=False))

    print("\n✨ FEEDBACK FINAL & SUGESTÕES PRÁTICAS")
    print("-" * 90)
    print(res_f4.get("Feedback_Final", {}).get("texto", "Não foi possível gerar o feedback final."))

    print("\n" + "="*90)
    print("✅ EXECUÇÃO COMPLETA DO AGENTE DE CARREIRA ADAPTATIVA FINALIZADA! (Saídas em JSON conforme requisito)")
    print("="*90 + "\n")


# =======================================================
# 10️⃣ - IMPRIME RELATÓRIO FINAL VISUAL
# =======================================================
print_relatorio_final(res_f1, res_f2, res_f3, res_f4)


---
## ✅ Como usar o notebook:

1. **Execute a célula de configuração inicial.**  
2. **Em seguida, execute todas as células (F1 → F4).**  
3. O notebook mostrará o relatório final com as respostas estruturadas em JSON e o feedback da simulação de entrevista.

💡 **Dica:** Para o vídeo de demonstração, utilize os dados já preenchidos no código (profissão, transição e respostas simuladas).
